![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 5. Mise en page & Application multi‑pages</b>

Dans le notebook précédent, vous avez rendu votre application **interactive** grâce aux widgets et au `session_state` : vos utilisateurs peuvent filtrer les données et voir les graphiques se mettre à jour en temps réel. Mais pour l'instant, tout tient sur une seule page — et quand le contenu s'allonge, ça devient vite difficile à lire.

Dans ce notebook, nous allons structurer l'application en **plusieurs pages distinctes**, chacune avec son propre rôle (KPI, exploration, à propos…). Nous verrons aussi comment organiser le contenu à l'intérieur d'une page grâce aux **colonnes** et aux **onglets**, deux outils de mise en page essentiels pour obtenir un rendu clair et professionnel.

# 0. Passer en application multi‑pages

Jusqu'à présent, tout le code de votre application vit dans un seul fichier `app.py`. Cela fonctionne, mais dès que vous ajoutez de nouvelles visualisations ou fonctionnalités, la page devient très longue et l'expérience utilisateur se dégrade. La solution : **découper votre application en plusieurs pages**, exactement comme les onglets d'un site web classique.

## 0.1. Le principe : un fichier = une page

La convention est très simple. Il vous suffit de créer un dossier `pages/` à côté de votre `app.py`, puis d'y déposer un fichier `.py` par page. Streamlit détecte automatiquement ces fichiers et génère un **menu de navigation** dans la barre latérale — vous n'avez aucun routeur à configurer.

```
streamlit_app/
│
├── app.py              ← page d'accueil
└── pages/
    ├── KPI.py          ← accessible via le menu latéral
    ├── Exploration.py
    └── A_propos.py
```

Le nom du fichier détermine le nom affiché dans le menu. Par exemple, `KPI.py` apparaîtra sous le libellé « KPI ». Vous pouvez aussi préfixer vos fichiers avec des numéros (`01_KPI.py`, `02_Exploration.py`) pour maîtriser l'ordre d'affichage.

## 0.2. Les méthodes de mise en page

Une fois vos pages créées, il vous faut organiser le contenu **à l'intérieur** de chacune d'elles. Streamlit propose pour cela plusieurs méthodes de layout que nous allons utiliser dans ce notebook.

**[`st.columns(n)`](https://docs.streamlit.io/develop/api-reference/layout/st.columns)** découpe la page en `n` colonnes côte à côte. C'est idéal pour afficher des KPI ou des métriques en un coup d'œil, comme vous le feriez dans un tableau de bord Excel ou PowerBI. Vous récupérez autant de variables que de colonnes, puis vous écrivez dans chacune d'elles avec un bloc `with` :

```python
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("CA total", "120 k€")
```

**[`st.tabs(["Onglet 1", "Onglet 2"])`](https://docs.streamlit.io/develop/api-reference/layout/st.tabs)** crée des onglets cliquables sur une même page. C'est très pratique quand vous voulez regrouper plusieurs visualisations alternatives sans surcharger l'affichage — l'utilisateur choisit simplement l'onglet qui l'intéresse.

**[`st.container()`](https://docs.streamlit.io/develop/api-reference/layout/st.container)** permet de regrouper des éléments dans un bloc logique. C'est surtout utile pour structurer votre code quand vous avez besoin d'insérer du contenu dans un ordre différent de celui dans lequel vous l'écrivez.

💡 Si vous venez de Jupyter, pensez aux colonnes et onglets comme un équivalent visuel des sous‑sections de notebook, mais directement dans l'interface web que vos utilisateurs verront.

## 0.3. Passons à la pratique

Nous allons créer trois pages pour notre application : une page **KPI** avec des indicateurs affichés en colonnes, une page **Exploration** avec des graphiques répartis dans des onglets, et une page **À propos**. Exécutez les cellules suivantes pour générer ces fichiers.

📖 [Documentation officielle — Applications multipages](https://docs.streamlit.io/develop/concepts/multipage-apps)

### Création de la page KPI

Cette première page utilise `st.columns` pour afficher trois indicateurs côte à côte, puis un graphique en dessous. Remarquez la syntaxe : nous créons trois variables (`col1`, `col2`, `col3`) et nous écrivons dans chacune avec `with`.

In [1]:
%%writefile ../streamlit_app/pages/KPI.py
import streamlit as st
import pandas as pd
from utils.data import load_data, filter_data
from utils.charts import make_line

# Titre de la page
st.title("📈 KPI")

# Chargement des données
data = load_data()

# Affichage des indicateurs clés dans 3 colonnes
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Total lignes", f"{len(data):,}".replace(",", " "))
with col2:
    st.metric("Dates", f"{data['date'].min().date()} → {data['date'].max().date()}")
with col3:
    st.metric("Catégories", ", ".join(map(str, data['categorie'].cat.categories)))

# Affichage de la tendance globale sous forme de courbe
st.subheader("Tendance globale")
st.plotly_chart(
    make_line(data, "date", "ventes", "categorie", "Ventes — global"),
    use_container_width=True
)


Writing ../streamlit_app/pages/KPI.py


### Création de la page Exploration

Cette page met en œuvre `st.tabs` pour proposer deux visualisations dans des onglets séparés : un scatter plot et un boxplot. L'utilisateur pourra passer de l'un à l'autre sans quitter la page.

In [2]:
%%writefile ../streamlit_app/pages/Exploration.py
import streamlit as st
import pandas as pd
import plotly.express as px
from utils.data import load_data

# Titre de la page
st.title("🔎 Exploration")

# Chargement des données
df = load_data()

# Création de deux onglets pour différentes visualisations
tab1, tab2 = st.tabs(["Scatter", "Boxplot"])

with tab1:
    # Affichage d'un nuage de points (scatter plot)
    fig = px.scatter(df, x="date", y="ventes", color="categorie", title="Dispersion")
    st.plotly_chart(fig, use_container_width=True)

with tab2:
    # Affichage d'un boxplot par catégorie
    fig = px.box(df, x="categorie", y="ventes", title="Répartition par catégorie")
    st.plotly_chart(fig, use_container_width=True)


Writing ../streamlit_app/pages/Exploration.py


### Création de la page À propos

Une page simple, mais toujours utile pour identifier l'application. En contexte professionnel, c'est là que vous placerez les informations sur la source des données, l'équipe, ou les conditions d'utilisation.

In [3]:
%%writefile ../streamlit_app/pages/A_propos.py
import streamlit as st

# Titre de la page "À propos"
st.title("ℹ️ À propos")

# Texte de présentation de l'application
st.write("Application pédagogique Streamlit + Plotly — DATAGONG")


Writing ../streamlit_app/pages/A_propos.py


⚠️ **Où sont passés mes filtres ?** Si vous relancez l'application et naviguez vers la page KPI ou Exploration, vous remarquerez que la sidebar avec les filtres (catégories, dates) a disparu. C'est tout à fait normal : en multi‑pages, chaque fichier est un script autonome. Quand vous ouvrez `KPI.py`, c'est **ce fichier seul** qui s'exécute — `app.py` n'est plus en jeu, donc les widgets qu'il définit ne sont pas rendus. Si vous souhaitez que les filtres apparaissent sur toutes les pages, il faudra les définir dans chaque page (en extrayant la logique dans une fonction utilitaire, par exemple). Les **valeurs** stockées dans `st.session_state` restent toutefois accessibles depuis n'importe quelle page — seul l'affichage des widgets disparaît.

# 1. Navigation & paramètres d'URL

Vos pages sont en place. Si vous relancez l'application maintenant, vous verrez le menu de navigation apparaître dans la barre latérale — Streamlit a tout détecté automatiquement. Voyons maintenant comment aller un cran plus loin en exploitant les **paramètres d'URL**.

## 1.1. La navigation entre les pages

Il n'y a rien à coder pour la navigation : dès que vous ajoutez un fichier dans `pages/`, Streamlit l'affiche dans le menu latéral. Le nom du fichier (sans l'extension `.py`) sert de libellé. C'est aussi simple que cela.

## 1.2. Synchroniser les filtres avec l'URL grâce à `st.query_params`

Vous connaissez peut-être les URL du type `https://monapp.com?categorie=A&annee=2025`. La partie après le `?` contient des **paramètres** (aussi appelés *query string*). Dans une application web classique, ces paramètres permettent de partager un lien qui pointe vers une vue précise — et Streamlit sait faire la même chose.

Concrètement, [`st.query_params`](https://docs.streamlit.io/develop/api-reference/utilities/st.query_params) vous permet de **lire** et **écrire** ces paramètres. Dans le code ci-dessous, nous allons écrire les valeurs de nos filtres (catégories sélectionnées, dates) directement dans l'URL. Résultat : quand vous modifiez un filtre, l'URL se met à jour en temps réel dans la barre d'adresse de votre navigateur.

L'intérêt est concret : vous pouvez **copier cette URL et la partager** à un collègue. En l'ouvrant, il retrouvera exactement les mêmes filtres appliqués, sans avoir à les re-sélectionner manuellement.

💡 Pour tester : une fois l'application relancée, modifiez un filtre dans la sidebar et observez l'URL dans votre navigateur — elle se met à jour automatiquement. Vous pouvez aussi essayer d'ajouter manuellement `?categorie=A` à la fin de l'URL et recharger la page.

⚠️ `st.query_params` ne remplace pas `st.session_state`. Les paramètres d'URL servent à partager un état *entre utilisateurs* via un lien, tandis que `session_state` gère l'état *au sein d'une même session*. Les deux sont complémentaires.

📖 [Documentation — `st.query_params`](https://docs.streamlit.io/develop/api-reference/utilities/st.query_params)

In [4]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Paramètres d'URL ---

# Écrit les filtres courants dans l'URL (query string)
# L'URL du navigateur se met à jour automatiquement
st.query_params["categorie"] = ",".join(f_cats)
st.query_params["date_min"] = str(dmin)
st.query_params["date_max"] = str(dmax)

# Affiche les paramètres d'URL pour vérifier
st.caption(f"🔗 Paramètres d'URL : {dict(st.query_params)}")


Appending to ../streamlit_app/app.py


# <font color='#ff7373'><b>Félicitations !</b></font>

Vous avez transformé une application monolithique en une **vraie application multi‑pages** avec une mise en page structurée — colonnes, onglets et navigation automatique. C'est un cap important : votre projet ressemble désormais à un outil professionnel.

Dans le prochain notebook, nous irons encore plus loin en construisant une page **Dashboard** complète avec des KPI et des tableaux de données.

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>